In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

np.random.seed(42)

dates = pd.date_range(start="2022-01-01", periods=730, freq="D")

seasonal = 20 * np.sin(2 * np.pi * dates.dayofyear / 7)
trend = np.linspace(50, 120, len(dates))

region_a = trend + seasonal + np.random.normal(0, 5, len(dates))
region_b = trend * 0.9 + seasonal + np.random.normal(0, 6, len(dates))
region_c = trend * 0.8 + seasonal + np.random.normal(0, 4, len(dates))
region_d = trend * 1.1 + seasonal + np.random.normal(0, 5, len(dates))

national = region_a + region_b + region_c + region_d

data = pd.DataFrame({
    "Region_A": region_a,
    "Region_B": region_b,
    "Region_C": region_c,
    "Region_D": region_d,
    "National": national
}, index=dates)

print(data.head())

train_size = int(len(data) * 0.8)
train = data.iloc[:train_size]
test = data.iloc[train_size:]

plot_acf(train["National"], lags=30)
plot_pacf(train["National"], lags=30)
plt.show()

model = SARIMAX(
    train["National"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

model_fit = model.fit()
print(model_fit.summary())


forecast_steps = len(test)
forecast_ind = model_fit.forecast(steps=forecast_steps)

region_forecasts = {}

for region in ["Region_A", "Region_B", "Region_C", "Region_D"]:
    m = SARIMAX(
        train[region],
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    f = m.fit()
    region_forecasts[region] = f.forecast(steps=forecast_steps)

reconciled_forecast = (
    region_forecasts["Region_A"]
    + region_forecasts["Region_B"]
    + region_forecasts["Region_C"]
    + region_forecasts["Region_D"]
)

rmse_ind = np.sqrt(mean_squared_error(test["National"], forecast_ind))
mape_ind = mean_absolute_percentage_error(test["National"], forecast_ind)

rmse_rec = np.sqrt(mean_squared_error(test["National"], reconciled_forecast))
mape_rec = mean_absolute_percentage_error(test["National"], reconciled_forecast)

print("Independent Forecast:")
print("RMSE:", rmse_ind)
print("MAPE:", mape_ind)

print("\nReconciled Forecast:")
print("RMSE:", rmse_rec)
print("MAPE:", mape_rec)

final_steps = 30

final_forecasts = {}

for region in ["Region_A", "Region_B", "Region_C", "Region_D"]:
    m = SARIMAX(
        data[region],
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    f = m.fit()
    final_forecasts[region] = f.forecast(steps=final_steps)

final_national = (
    final_forecasts["Region_A"]
    + final_forecasts["Region_B"]
    + final_forecasts["Region_C"]
    + final_forecasts["Region_D"]
)

final_output = pd.DataFrame(final_forecasts)
final_output["National"] = final_national

print("\nFINAL 30-STEP FORECAST:")
print(final_output)
